In [ ]:
!pip uninstall -y transformers
!pip install transformers==4.51.3 accelerate qwen-vl-utils 

In [2]:
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

In [3]:
!pip install wikipedia-api requests sentence-transformers faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 91.9 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [4]:
!pip install gliner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 83.6 MB/s eta 0:00:00:00:0100:01


# FULL FINAL 

In [5]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch
from gliner import GLiNER
import json
import re
import requests
import numpy as np
from typing import Optional, List, Dict, Any
from sentence_transformers import SentenceTransformer
import faiss
import wikipediaapi

MODEL_NAME  = "Qwen/Qwen2-VL-2B-Instruct"
EMBED_MODEL = "all-MiniLM-L6-v2"

OPENFDA_BASE    = "https://api.fda.gov/drug/label.json"
RXNORM_BASE     = "https://rxnav.nlm.nih.gov/REST"
WIKI_USER_AGENT = "MedicalOCR-RAG/1.0 (research use)"

GLINER_LABELS = ["drug", "strength", "dosage", "frequency", "duration", "form", "route"]

REACT_MAX_STEPS = 6   # max agent iterations before forced answer



2026-05-23 05:34:09.260017: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779514449.454430     119 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779514449.510128     119 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779514449.981704     119 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779514449.981735     119 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779514449.981738     119 computation_placer.cc:177] computation placer alr

In [6]:

# =========================================
# OCR Module
# =========================================
class HandwritingOCR:
    def __init__(self, model_name=MODEL_NAME):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = Qwen2VLForConditionalGeneration.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device_map="auto",
        )
        self.processor = AutoProcessor.from_pretrained(model_name)

    def extract_text(self, image_path: str, max_new_tokens: int = 512) -> str:
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": (
                    "Extract all handwritten text from this image. "
                    "Correct obvious OCR spelling mistakes. "
                    "Preserve line breaks and formatting. "
                    "Return ONLY the final cleaned text."
                )},
            ],
        }]
        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = self.processor(
            text=[text], images=image_inputs, videos=video_inputs,
            padding=True, return_tensors="pt",
        ).to(self.device)

        generated_ids = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
        output_text = self.processor.batch_decode(trimmed, skip_special_tokens=True,
                                                   clean_up_tokenization_spaces=True)[0]
        for prefix in ["Here is the extracted text:", "Extracted text:",
                        "The handwritten text is:", "Output:", "Answer:"]:
            if output_text.startswith(prefix):
                output_text = output_text[len(prefix):].strip()
        return output_text.strip()



In [7]:

# =========================================
# Tool Definitions (callable by the agent)
# =========================================
class MedicalTools:
    def __init__(self, retriever):
        self.retriever = retriever
        self.wiki = wikipediaapi.Wikipedia(language="en", user_agent=WIKI_USER_AGENT)

    def search_knowledge_base(self, query: str, top_k: int = 5) -> str:
        passages = self.retriever.retrieve(query, top_k=top_k)
        if not passages:
            return "No matching records found in internal indexes."
        return "\n\n".join([f"[{i+1}] (Source: {p['source']})\n{p['text']}" for i, p in enumerate(passages)])

    def fetch_openfda_field(self, drug_name: str, field: str) -> str:
        valid_fields = {"adverse_reactions", "warnings", "contraindications", "drug_interactions", "dosage_and_administration", "indications_and_usage", "description"}
        field = field.strip().lower().replace(" ", "_")
        if field not in valid_fields:
            return f"Invalid field choice '{field}'."
        passages = self.retriever._fetch_openfda(drug_name)
        if not passages:
            # Nothing in OpenFDA — fall back to the knowledge base
            kb = self.search_knowledge_base(f"{drug_name} {field.replace('_', ' ')}", top_k=5)
            return kb if kb else f"No OpenFDA data found for '{drug_name}'."
        matched = [p for p in passages if p.get("field") == field]
        if not matched:
            matched = passages  # return all fields if specific one absent
        return "\n\n".join(
            f"OpenFDA {p.get('field', field)} for {drug_name}:\n{p['text']}"
            for p in matched[:3]
        )

    def fetch_wikipedia(self, drug_name: str) -> str:
        for term in [drug_name, f"{drug_name} (medication)"]:
            page = self.wiki.page(term)
            if page.exists():
                return f"Wikipedia ({page.title}):\n{page.summary[:600]}"
        return f"No Wikipedia entry for '{drug_name}'."

    def lookup_rxnorm(self, drug_name: str) -> str:
        canonical = self.retriever.rxnorm_canonical(drug_name)
        return f"RxNorm Canonical Name: {canonical}" if canonical else "No matching RxNorm record."

    def load_drug_knowledge(self, drug_name: str) -> str:
        self.retriever.load_for_entities([drug_name])
        return f"Loaded real-time references for '{drug_name}'."

    def web_search_engine(self, query: str) -> str:
        """Fallback tool allowing agent to explore generic web sources."""
        try:
            with DDGS() as ddgs:
                results = [r for r in ddgs.text(query, max_results=3)]
            if not results:
                return "No web matches found."
            return "\n\n".join([f"URL: {r['href']}\nSnippet: {r['body']}" for r in results])
        except Exception as e:
            return f"Search engine connection error: {e}"

    def dispatch(self, tool_name: str, args: Dict[str, str]) -> str:
        tool_map = {
            "search_knowledge_base": lambda a: self.search_knowledge_base(a.get("query", ""), int(a.get("top_k", 5))),
            "fetch_openfda_field":   lambda a: self.fetch_openfda_field(a.get("drug_name", ""), a.get("field", "adverse_reactions")),
            "fetch_wikipedia":       lambda a: self.fetch_wikipedia(a.get("drug_name", "")),
            "lookup_rxnorm":         lambda a: self.lookup_rxnorm(a.get("drug_name", "")),
            "load_drug_knowledge":   lambda a: self.load_drug_knowledge(a.get("drug_name", "")),
            "web_search_engine":     lambda a: self.web_search_engine(a.get("query", "")),
        }
        fn = tool_map.get(tool_name)
        if fn is None: return f"Unknown tool: {tool_name}"
        try: return fn(args)
        except Exception as e: return f"Execution error: {e}"

# =========================================
# Qwen2-VL Inference Helper
# =========================================
class QwenInference:
    """Thin wrapper so any module can call the shared Qwen model."""

    def __init__(self, model, processor):
        self.model = model
        self.processor = processor
        self.device = next(model.parameters()).device

    def generate(self, prompt: str, max_new_tokens: int = 512) -> str:
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
        inputs = self.processor(text=[text], return_tensors="pt").to(self.device)
        with torch.no_grad():
            ids = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        trimmed = ids[0][inputs.input_ids.shape[1]:]
        return self.processor.decode(trimmed, skip_special_tokens=True).strip()




In [8]:
# =========================================
# FIXED RE-ACT AGENT
# =========================================

REACT_SYSTEM = """You are a precise medical reasoning assistant with access to tools.
You operate in a strict Thought → Action → Args loop.
Stop generating immediately after writing Args. Do NOT write the Observation yourself.

Available tools:
1. load_drug_knowledge  - ALWAYS call this first for any drug to fetch fresh data.
   Args: {"drug_name": "name"}
2. search_knowledge_base - Search loaded passages for information.
   Args: {"query": "text", "top_k": "5"}
3. fetch_openfda_field  - Retrieve a specific label section from OpenFDA.
   Choices: adverse_reactions, warnings, contraindications,
            drug_interactions, dosage_and_administration, indications_and_usage
   Args: {"drug_name": "name", "field": "field_name"}
4. fetch_wikipedia      - Get drug summary from Wikipedia.
   Args: {"drug_name": "name"}
5. lookup_rxnorm        - Get canonical drug name from RxNorm.
   Args: {"drug_name": "name"}
6. finish               - Return the final answer to the user.
   Args: {"answer": "Your complete synthesis here"}

Strict output format (no deviations):
Thought: <your reasoning>
Action: <tool_name>
Args: {"key": "value"}
"""

REACT_FEW_SHOT = """
Example:
Question: What are the side effects of Amoxicillin?
Thought: I need to load knowledge about Amoxicillin first.
Action: load_drug_knowledge
Args: {"drug_name": "Amoxicillin"}
Observation: Loaded real-time references for 'Amoxicillin'.
Thought: Now I'll fetch the adverse reactions from OpenFDA.
Action: fetch_openfda_field
Args: {"drug_name": "Amoxicillin", "field": "adverse_reactions"}
Observation: OpenFDA adverse_reactions for Amoxicillin: Nausea, vomiting, diarrhea, rash...
Thought: I have enough information.
Action: finish
Args: {"answer": "Common side effects of Amoxicillin include nausea, vomiting, diarrhea, and rash."}
"""


class ReActAgent:
    def __init__(self, qwen: "QwenInference", tools: "MedicalTools"):
        self.qwen = qwen
        self.tools = tools

    # ------------------------------------------------------------------
    # Parsing
    # ------------------------------------------------------------------
    @staticmethod
    def _parse_action(text: str):
        """Extract the FIRST Action/Args block; ignore everything after."""
        action_m = re.search(r"Action:\s*([A-Za-z0-9_]+)", text)
        args_m   = re.search(r"Args:\s*(\{.*?\})", text, re.DOTALL)
        if not action_m:
            return None, {}
        action = action_m.group(1).strip()
        args   = {}
        if args_m:
            raw = args_m.group(1).strip()
            try:
                args = json.loads(raw)
            except Exception:
                try:
                    args = eval(raw)
                except Exception:
                    pass
        return action, args

    # ------------------------------------------------------------------
    # Prompt builder
    # ------------------------------------------------------------------
    @staticmethod
    def _build_prompt(question: str, drug_names: list, trajectory: list) -> str:
        prompt = f"{REACT_SYSTEM}\n{REACT_FEW_SHOT}\n"
        if drug_names:
            prompt += (
                f"Context: The prescription contains these drug(s): "
                f"{', '.join(drug_names)}\n\n"
            )
        prompt += f"Current Task:\nQuestion: {question}\n\n"
        for step in trajectory:
            prompt += f"Thought: {step['thought']}\n"
            prompt += f"Action: {step['action']}\n"
            prompt += f"Args: {json.dumps(step['args'])}\n"
            if step["action"] != "finish":
                prompt += f"Observation: {step['observation']}\n\n"
        prompt += "Thought:"
        return prompt

    # ------------------------------------------------------------------
    # Forced warm-up: pre-load knowledge for every known drug so the
    # retriever is never empty when the first real query fires.
    # ------------------------------------------------------------------
    def _preload_drugs(self, drug_names: list, trajectory: list):
        already_loaded = {
            s["args"].get("drug_name")
            for s in trajectory
            if s["action"] == "load_drug_knowledge"
        }
        for drug in drug_names:
            if drug and drug not in already_loaded:
                print(f"[ReAct] Pre-loading knowledge for '{drug}'…")
                obs = self.tools.dispatch("load_drug_knowledge", {"drug_name": drug})
                trajectory.append({
                    "thought": f"Pre-loading external references for {drug}.",
                    "action":  "load_drug_knowledge",
                    "args":    {"drug_name": drug},
                    "observation": obs,
                })

    # ------------------------------------------------------------------
    # Heuristic: map question intent → best first tool call
    # ------------------------------------------------------------------
    @staticmethod
    def _heuristic_first_action(question: str, drug_names: list):
        q = question.lower()
        if not drug_names:
            return None, {}
        drug = drug_names[0]
        field_map = {
            ("side effect", "adverse", "reaction", "risk"): "adverse_reactions",
            ("warning", "caution", "precaution"):           "warnings",
            ("contraindic", "avoid"):                       "contraindications",
            ("interact", "combination", "together"):        "drug_interactions",
            ("dose", "dosage", "how much", "how many", "administration"): "dosage_and_administration",
            ("use", "treat", "indicat", "purpose", "for what"): "indications_and_usage",
        }
        for keywords, field in field_map.items():
            if any(kw in q for kw in keywords):
                return "fetch_openfda_field", {"drug_name": drug, "field": field}
        # Default: search the already-loaded knowledge base
        return "search_knowledge_base", {"query": question, "top_k": "5"}

    # ------------------------------------------------------------------
    # Main loop
    # ------------------------------------------------------------------
    def run(self, question: str, drug_names: list = None) -> dict:
        drug_names = drug_names or []
        trajectory: list = []

        # ── Step 0: guarantee knowledge is indexed before the loop ──
        self._preload_drugs(drug_names, trajectory)

        final_answer = None

        for step_num in range(1, REACT_MAX_STEPS + 1):
            print(f"[ReAct] Step {step_num}/{REACT_MAX_STEPS}")
            prompt = self._build_prompt(question, drug_names, trajectory)

            raw = self.qwen.generate(prompt, max_new_tokens=300)

            # Strip hallucinated Observations
            if "Observation:" in raw:
                raw = raw.split("Observation:")[0].strip()

            print(f"[ReAct] LLM raw:\n{raw}\n")

            # Extract thought
            thought = "Reasoning about the question."
            if "Action:" in raw:
                thought = raw.split("Action:")[0].replace("Thought:", "").strip()
            elif "Thought:" in raw:
                thought = raw.replace("Thought:", "").strip()

            action, args = self._parse_action(raw)

            # ── Fallback when model fails to produce a valid action ──
            if action is None:
                # First real step: use heuristic
                action, args = self._heuristic_first_action(question, drug_names)
                if action is None:
                    # Nothing we can do; wrap up with whatever we have
                    obs_texts = " ".join(
                        s["observation"][:300]
                        for s in trajectory
                        if s.get("observation")
                    )
                    final_answer = (
                        obs_texts if obs_texts.strip()
                        else "I could not find specific information for this question."
                    )
                    break
                thought = f"Heuristic: applying {action} based on question intent."

            # ── Finish ──────────────────────────────────────────────
            if action.lower() == "finish":
                final_answer = args.get("answer", thought)
                trajectory.append({
                    "thought": thought, "action": "finish",
                    "args": args, "observation": "",
                })
                break

            # ── Execute tool ─────────────────────────────────────────
            observation = self.tools.dispatch(action, args)
            print(f"[ReAct] '{action}' → {observation[:200]}…")
            trajectory.append({
                "thought": thought,
                "action":  action,
                "args":    args,
                "observation": observation,
            })

        else:
            # Max steps exhausted → synthesise from all collected observations
            print("[ReAct] Max steps hit — synthesising from collected observations…")
            obs_summary = "\n".join(
                f"- {s['action']}: {s['observation'][:400]}"
                for s in trajectory
                if s.get("observation")
            )
            if obs_summary.strip():
                synth_prompt = (
                    f"{REACT_SYSTEM}\n\n"
                    f"Using ONLY the factual findings below, write a clear, complete "
                    f"answer to: '{question}'\n\n"
                    f"Findings:\n{obs_summary}\n\n"
                    f"Action: finish\nArgs: "
                )
                raw_synth = self.qwen.generate(synth_prompt, max_new_tokens=400)
                _, synth_args = self._parse_action(raw_synth)
                final_answer = synth_args.get("answer") or obs_summary
            else:
                final_answer = "I could not retrieve enough information to answer this question."

        # ── Source attribution ────────────────────────────────────────
        sources = set()
        for step in trajectory:
            if step["action"] == "fetch_openfda_field":
                sources.add(f"OpenFDA: {step['args'].get('drug_name')}")
            if step["action"] == "fetch_wikipedia":
                sources.add(f"Wikipedia: {step['args'].get('drug_name')}")
            if step["action"] == "load_drug_knowledge":
                sources.add(f"Loaded: {step['args'].get('drug_name')}")

        return {
            "question":    question,
            "answer":      final_answer,
            "trajectory":  trajectory,
            "sources":     list(sources),
            "steps_taken": len(trajectory),
        }


In [12]:
from rank_bm25 import BM25Okapi
from duckduckgo_search import DDGS
import numpy as np
import faiss
import requests

class MedicalKnowledgeRetriever:
    def __init__(self, embed_model: str = EMBED_MODEL):
        self.embedder = SentenceTransformer(embed_model)
        self.wiki = wikipediaapi.Wikipedia(language="en", user_agent=WIKI_USER_AGENT)
        self._passages: List[dict] = []
        self._index: Optional[faiss.IndexFlatIP] = None
        self._bm25 = None
        self._embeddings = None

    def _fetch_web_fallback(self, query: str) -> List[dict]:
        """Queries live web engine search when database structures return empty results."""
        chunks = []
        try:
            print(f"[Web Search] Querying fallbacks for: '{query}'")
            with DDGS() as ddgs:
                results = [r for r in ddgs.text(query, max_results=4)]
                
            for res in results:
                body = res.get("body", "")
                title = res.get("title", "Reference Document")
                href = res.get("href", "URL")
                
                if len(body.split()) > 8:
                    chunks.append({
                        "text": f"[{title}] {body}",
                        "source": f"Web: {href}",
                        "entity": query
                    })
        except Exception as e:
            print(f"[Web Search] Fallback connection skip: {e}")
        return chunks

    def _fetch_wikipedia(self, term: str) -> List[dict]:
        page = self.wiki.page(term)
        if not page.exists():
            page = self.wiki.page(f"{term} (medication)")
        if not page.exists():
            return []
        words = page.summary.split()
        chunks = []
        for i in range(0, min(len(words), 600), 80):
            chunk = " ".join(words[i:i+80])
            if chunk:
                chunks.append({
                    "text": chunk,
                    "source": f"Wikipedia: {page.title}",
                    "entity": term
                })
        return chunks

    def _fetch_openfda(self, drug_name: str) -> List[dict]:
        passages = []
        for search_type in ["generic_name", "brand_name"]:
            try:
                resp = requests.get(
                    OPENFDA_BASE,
                    params={"search": f'openfda.{search_type}:"{drug_name}"', "limit": 1},
                    timeout=8
                )
                result = resp.json().get("results", [])
                if result:
                    label = result[0]
                    for field in ["description", "indications_and_usage", "warnings", 
                                  "dosage_and_administration", "adverse_reactions", 
                                  "drug_interactions", "contraindications"]:
                        content = label.get(field, [])
                        if content:
                            passages.append({
                                "text": " ".join(content)[:1200],
                                "source": f"OpenFDA: {drug_name}",
                                "entity": drug_name,
                                "field": field
                            })
                    break
            except Exception:
                pass
        return passages

    def rxnorm_canonical(self, drug_name: str) -> Optional[str]:
        try:
            resp = requests.get(f"{RXNORM_BASE}/drugs.json", params={"name": drug_name}, timeout=6)
            data = resp.json()
            for group in data.get("drugGroup", {}).get("conceptGroup", []):
                for concept in group.get("conceptProperties", []):
                    if concept.get("name"):
                        return concept["name"]
        except Exception:
            pass
        return None

    def _build_index(self, passages: List[dict]):
        if not passages:
            return
        self._passages = list(passages)
        texts = [p["text"] for p in passages]
        
        # Dense Embedding Index Setup
        self._embeddings = self.embedder.encode(texts, normalize_embeddings=True, convert_to_numpy=True).astype("float32")
        self._index = faiss.IndexFlatIP(self._embeddings.shape[1])
        self._index.add(self._embeddings)
        
        # Sparse BM25 Keyword Setup
        tokenized = [t.lower().split() for t in texts]
        self._bm25 = BM25Okapi(tokenized)

    def retrieve(self, query: str, top_k: int = 4) -> List[dict]:
        if self._index is None or not self._passages:
            return []

        # 1. Dense Semantic Search Mapping
        q_emb = self.embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
        scores, indices = self._index.search(q_emb, top_k)
        dense_results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx >= len(self._passages) or idx < 0: 
                continue
            dense_results.append({**self._passages[idx], "score": float(score)})

        # 2. Sparse Search Fallback Check
        sparse_results = []
        if self._bm25 is not None:
            bm25_scores = self._bm25.get_scores(query.lower().split())
            sparse_indices = np.argsort(bm25_scores)[::-1][:top_k]
            for idx in sparse_indices:
                sparse_results.append({**self._passages[idx], "score": float(bm25_scores[idx])})

        # Hybrid Reciprocal Scoring Interleaving
        merged = {}
        for r in dense_results + sparse_results:
            key = r["text"]
            if key not in merged:
                merged[key] = r
            else:
                merged[key]["score"] += r["score"]
                
        return sorted(merged.values(), key=lambda x: x["score"], reverse=True)[:top_k]

    def load_for_entities(self, entities: List[str]):
        all_passages = []
        existing_entities = set(p.get("entity") for p in self._passages)

        for entity in entities:
            if entity in existing_entities:
                continue

            wiki_data = self._fetch_wikipedia(entity)
            fda_data = self._fetch_openfda(entity)
            
            all_passages.extend(wiki_data)
            all_passages.extend(fda_data)

            # Web backup activates if core clinical datasets don't match
            if not wiki_data and not fda_data:
                web_data = self._fetch_web_fallback(f"{entity} drug information effects dosage")
                all_passages.extend(web_data)

        if all_passages:
            combined = list(self._passages) + all_passages
            self._build_index(combined)
        print(f"[RAG] Base synced. Active passages count: {len(self._passages)}")

In [13]:

# =========================================
# NER Correction Module
# =========================================
class NERCorrector:
    def __init__(self, retriever: MedicalKnowledgeRetriever):
        self.retriever = retriever

    def correct_drug_name(self, drug_text: str) -> dict:
        canonical = self.retriever.rxnorm_canonical(drug_text)
        passages  = self.retriever.retrieve(drug_text, top_k=5)
        return {
            "original":        drug_text,
            "corrected":       canonical if canonical else drug_text,
            "rxnorm_found":    canonical is not None,
            "evidence_snippets": [p["text"][:200] for p in passages],
            "sources":         list({p["source"] for p in passages}),
        }

    def correct_entities(self, entities: List[dict]) -> List[dict]:
        corrected = []
        for ent in entities:
            if ent["label"] == "drug":
                correction = self.correct_drug_name(ent["text"])
                corrected.append({**ent, "correction": correction})
            else:
                passages = self.retriever.retrieve(
                    f"{ent['text']} {ent['label']} medication", top_k=2)
                corrected.append({**ent, "correction": {
                    "original":          ent["text"],
                    "corrected":         ent["text"],
                    "evidence_snippets": [p["text"][:200] for p in passages],
                    "sources":           list({p["source"] for p in passages}),
                }})
        return corrected


In [14]:
# =========================================
# Q&A Module  (now delegates to agent if retriever has data)
# =========================================
class MedicalQA:
    def __init__(self, retriever: "MedicalKnowledgeRetriever"):
        self.retriever = retriever

    def answer(self, question: str, top_k: int = 5) -> dict:
        passages = self.retriever.retrieve(question, top_k=top_k)
        if not passages:
            return {
                "question": question,
                "answer": (
                    "No relevant information found in the loaded knowledge base. "
                    "Try loading knowledge for the relevant drug entities first."
                ),
                "sources": [],
            }

        # Collate the top passages into a readable answer
        context = "\n\n".join(
            f"[{i+1}] ({p['source']})\n{p['text']}"
            for i, p in enumerate(passages)
        )
        answer_text = (
            f"Based on retrieved medical sources:\n\n{context}"
        )
        return {
            "question": question,
            "answer":   answer_text,
            "context_passages": [
                {"source": p["source"], "text": p["text"],
                 "relevance_score": p.get("score", 0)}
                for p in passages
            ],
            "sources": list({p["source"] for p in passages}),
        }


In [15]:
# =========================================
# Main Pipeline
# =========================================
class MedicalPipeline:
    def __init__(self, ocr_model: str = MODEL_NAME, embed_model: str = EMBED_MODEL):
        print("[Pipeline] Loading OCR / reasoning model (Qwen2-VL)...")
        self.ocr       = HandwritingOCR(ocr_model)
        self.qwen      = QwenInference(self.ocr.model, self.ocr.processor)

        print("[Pipeline] Loading GLiNER NER model...")
        self.ner_model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")

        print("[Pipeline] Initialising retriever + tools...")
        self.retriever = MedicalKnowledgeRetriever(embed_model)
        self.corrector = NERCorrector(self.retriever)

        self.tools  = MedicalTools(self.retriever)
        self.agent  = ReActAgent(self.qwen, self.tools)
        self.qa     = MedicalQA(self.retriever)

    # ---- OCR ----
    def run_ocr(self, image_path: str) -> str:
        print(f"[OCR] Processing {image_path}...")
        text = self.ocr.extract_text(image_path)
        print(f"[OCR] Result:\n{text}\n")
        return text

    # ---- NER ----
    def run_ner(self, text: str) -> list:
        entities = self.ner_model.predict_entities(text, labels=GLINER_LABELS, threshold=0.4)
        formatted = [{
            "text":  e["text"],
            "label": e["label"].lower(),
            "start": e["start"],
            "end":   e["end"],
            "score": round(e["score"], 4),
        } for e in entities]
        print(f"[NER] Found {len(formatted)} entities:")
        for e in formatted:
            print(f"  - {e['text']} ({e['label']}) [score={e['score']}]")
        return formatted

    # ---- Knowledge loading ----
    def load_knowledge(self, entities: list):
        drug_terms = list({e["text"] for e in entities if e["label"] == "drug"})
        if not drug_terms:
            print("[RAG] No DRUG entities found — skipping knowledge load.")
            return
        self.retriever.load_for_entities(drug_terms)

    # ---- NER correction ----
    def run_correction(self, entities: list) -> list:
        corrected = self.corrector.correct_entities(entities)
        for e in corrected:
            c = e["correction"]
            if c["corrected"] != c["original"]:
                print(f"[Correction] '{c['original']}' → '{c['corrected']}' (RxNorm)")
            else:
                print(f"[Correction] '{c['original']}' — no correction needed.")
        return corrected

    # ---- Structured JSON ----
    def to_structured_json(self, corrected_entities: list) -> dict:
        result = {label: [] for label in GLINER_LABELS}
        for ent in corrected_entities:
            label = ent["label"]
            if label in result:
                display = ent["correction"]["corrected"] if label == "drug" else ent["text"]
                result[label].append({
                    "value":        display,
                    "original_ocr": ent["text"],
                    "sources":      ent["correction"].get("sources", []),
                })
        return result

    # ---- Full image pipeline ----
    def process_image(self, image_path: str) -> dict:
        text       = self.run_ocr(image_path)
        entities   = self.run_ner(text)
        self.load_knowledge(entities)          # always index before Q&A
        corrected  = self.run_correction(entities)
        structured = self.to_structured_json(corrected)
        return {
            "raw_text":                text,
            "entities_raw":            entities,
            "entities_corrected":      corrected,
            "structured_prescription": structured,
        }

    # ---- Text-only pipeline ----
    def process_text(self, text: str) -> dict:
        entities   = self.run_ner(text)
        self.load_knowledge(entities)
        corrected  = self.run_correction(entities)
        structured = self.to_structured_json(corrected)
        return {
            "raw_text":                text,
            "entities_raw":            entities,
            "entities_corrected":      corrected,
            "structured_prescription": structured,
        }

    # ---- Agentic Q&A (FIXED) ----
    def ask(self, question: str, structured_prescription: dict = None) -> dict:
        drug_names = []
        if structured_prescription:
            for entry in structured_prescription.get("drug", []):
                name = entry.get("value") or entry.get("original_ocr", "")
                if name:
                    drug_names.append(name)

        # ── KEY FIX: always ensure knowledge is loaded before the agent fires ──
        if drug_names:
            self.retriever.load_for_entities(drug_names)

        print(f"[Agent] Question : {question}")
        print(f"[Agent] Known drugs: {drug_names}")
        return self.agent.run(question, drug_names=drug_names)


In [ ]:


# =========================================
# Gradio UI
# =========================================
import gradio as gr
import pandas as pd
import tempfile
import os

pipeline = MedicalPipeline()


def run_pipeline(image, question):
    if image is None:
        return None, "No image uploaded.", pd.DataFrame(), "{}", "", "", ""

    temp_path = None
    try:
        with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
            temp_path = tmp.name
            image.save(tmp, format="PNG")

        result     = pipeline.process_image(temp_path)
        raw_text   = result.get("raw_text", "")

        # NER table
        ner_rows = []
        for ent in result.get("entities_corrected", []):
            c = ent.get("correction", {})
            ner_rows.append({
                "Label":      ent.get("label", ""),
                "Original":   c.get("original", ""),
                "Corrected":  c.get("corrected", ""),
                "Confidence": ent.get("score", ""),
            })
        ner_df = pd.DataFrame(ner_rows)

        structured_json = json.dumps(result.get("structured_prescription", {}), indent=2)

        answer_text  = ""
        source_text  = ""
        agent_trace  = ""
        if question.strip():
            qa_result   = pipeline.ask(question, result.get("structured_prescription"))
            answer_text = qa_result.get("answer", "")
            sources     = qa_result.get("sources", [])
            source_text = "\n".join(sources) if sources else "N/A"

            # Format the ReAct trajectory for display
            trace_lines = []
            for i, step in enumerate(qa_result.get("trajectory", []), 1):
                trace_lines.append(f"── Step {i} ──")
                if step.get("thought"):
                    trace_lines.append(f"Thought: {step['thought']}")
                trace_lines.append(f"Action:  {step['action']}({json.dumps(step.get('args', {}))})")
                if step.get("observation"):
                    obs_preview = step["observation"][:300]
                    trace_lines.append(f"Obs:     {obs_preview}{'...' if len(step['observation'])>300 else ''}")
            agent_trace = "\n".join(trace_lines)

        return image, raw_text, ner_df, structured_json, answer_text, source_text, agent_trace

    except Exception as e:
        import traceback
        return image, f"Error: {e}\n{traceback.format_exc()}", pd.DataFrame(), "{}", "", "", ""

    finally:
        if temp_path and os.path.exists(temp_path):
            try:
                os.unlink(temp_path)
            except Exception:
                pass


with gr.Blocks(title="Medical OCR + NER + Agentic RAG") as demo:
    gr.Markdown("""
    # Medical Prescription Pipeline — Agentic RAG (ReAct)
    Upload a prescription image and ask a question.
    The agent will reason step-by-step, call tools (OpenFDA, Wikipedia, RxNorm, FAISS),
    and synthesise a grounded answer — all locally using Qwen2-VL.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            image_input    = gr.Image(type="pil", label="Upload Prescription Image")
            question_input = gr.Textbox(
                label="Ask a Medical Question",
                placeholder="e.g. What are the side effects of the medicines prescribed?",
            )
            run_btn = gr.Button("Run Pipeline", variant="primary")

        with gr.Column(scale=1):
            image_preview = gr.Image(label="Uploaded Image")
            ocr_output    = gr.Textbox(label="Step 1: OCR Output", lines=8)

    ner_output = gr.Dataframe(
        headers=["Label", "Original", "Corrected", "Confidence"],
        label="Step 2: NER + Corrections",
        interactive=False,
    )

    structured_output = gr.Code(
        label="Step 3: Structured Prescription JSON", language="json")

    with gr.Row():
        answer_output = gr.Textbox(label="Step 4: Agent Answer", lines=8)
        source_output = gr.Textbox(label="Sources Used", lines=8)

    agent_trace_output = gr.Textbox(
        label="Agent Reasoning Trace (ReAct steps)", lines=20)

    # Ensure all elements match output parameters structurally
    run_btn.click(
        fn=run_pipeline,
        inputs=[image_input, question_input],
        outputs=[
            image_preview, 
            ocr_output, 
            ner_output, 
            structured_output,
            answer_output, 
            source_output, 
            agent_trace_output
        ]
    )

demo.launch(debug=True, share=True)


[Pipeline] Loading OCR / reasoning model (Qwen2-VL)...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/429M [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

[OCR] Result:
John Smith
34
162 Example St, NY
MEDICAL CENTRE
824 14th Street
New York, NY 91743, USA
NAME
AGE
RX
Betaloc 100mg - 1 tab BID
Dorzolamide 10 mg - 1 tab BID
Cimetidine 50 mg - 2 tabs TID
Oxperelol 50mg - 1 tab QD
Dr. Steve Johnson
signature
REFILL 0 1 2 3 4 5 PRN
LABEL
WTX-N-PRESCT-T
1-889-422-0700



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[NER] Found 5 entities:
  - Betaloc (drug) [score=0.7999]
  - Dorzolamide (drug) [score=0.861]
  - Cimetidine (drug) [score=0.8586]
  - Oxperelol (drug) [score=0.8166]
  - 50mg (dosage) [score=0.4555]


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [10]:
!pip install rank-bm25

In [11]:
!pip install duckduckgo-search -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 103.3 MB/s eta 0:00:0000:01
